# Structured Streaming — 05: End-to-End Pipeline

Complete e-commerce order processing pipeline.

## Architecture
```
  [CSV Files] ──► [File Source Stream]
                         │
                         ▼
                  [Enrich + Transform]  ◄──  [Product Lookup (Static)]
                         │
               ┌─────────┴──────────┐
               ▼                    ▼
       [Parquet Sink]       [Windowed Agg]
       (raw enriched)       (per-category)
                                    │
                                    ▼
                           [Memory Sink / Dashboard]
```

## What You Will Build
- **File watcher**: File source reads new CSV order files as they land in an input directory
- **Enrichment**: Stream-static broadcast join with a product catalog DataFrame
- **Dual sink**: Parquet sink (durable raw events) + Memory sink (live dashboard)

## Skills from Prior Notebooks
- **01**: File source, explicit schema, `maxFilesPerTrigger`
- **02**: Stream-static join, broadcast hints
- **03**: Windowing, watermark, `groupBy(window(...))`
- **04**: Multiple concurrent queries, `foreachBatch` pattern

## 1. Setup & Directory Layout

In [ ]:
import tempfile, os, time, threading, csv, random, shutil, json
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, window, count,
    sum as spark_sum, avg, when, broadcast, round as spark_round,
    input_file_name
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("SS-05-E2E-Pipeline")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE         = tempfile.mkdtemp().replace('\\', '/') + '/ss05'
INPUT_DIR    = BASE + '/orders_input'
PARQUET_OUT  = BASE + '/enriched_parquet'
CKPT_ENRICH  = BASE + '/ckpt_enriched'
CKPT_WINDOW  = BASE + '/ckpt_window'

for d in [INPUT_DIR, PARQUET_OUT, CKPT_ENRICH, CKPT_WINDOW]:
    os.makedirs(d, exist_ok=True)

print("Pipeline directories created:")
for name, path in [("Input (CSV watch)", INPUT_DIR), ("Parquet sink", PARQUET_OUT),
                   ("Ckpt enriched", CKPT_ENRICH), ("Ckpt window", CKPT_WINDOW)]:
    print(f"  {name}: {path}")

## 2. Static Reference Data — Product Catalog

In [ ]:
PRODUCT_DATA = [
    (1,  "Laptop",     1200.00, "Electronics"),
    (2,  "Mouse",        25.00, "Electronics"),
    (3,  "Keyboard",     75.00, "Electronics"),
    (4,  "Monitor",     400.00, "Electronics"),
    (5,  "Headset",      90.00, "Electronics"),
    (6,  "Desk",        300.00, "Furniture"),
    (7,  "Chair",       250.00, "Furniture"),
    (8,  "Lamp",         40.00, "Furniture"),
    (9,  "Notebook",      5.00, "Stationery"),
    (10, "Pen Set",      12.00, "Stationery"),
]
product_schema = StructType([
    StructField("product_id",   IntegerType(), False),
    StructField("product_name", StringType(),  False),
    StructField("unit_price",   DoubleType(),  False),
    StructField("category",     StringType(),  False),
])
products_df = spark.createDataFrame(PRODUCT_DATA, product_schema)
print("Product catalog:")
products_df.show()

## 3. Order Schema & CSV Generator

In [ ]:
order_schema = StructType([
    StructField("order_id",    IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id",  IntegerType(), True),
    StructField("quantity",    IntegerType(), True),
])

CUSTOMERS = list(range(1001, 1011))   # 10 customers

def write_order_batch(batch_num):
    path = os.path.join(INPUT_DIR, f"orders_{batch_num:04d}.csv")
    rows = []
    for i in range(8):   # 8 orders per file
        rows.append([
            batch_num * 100 + i,
            random.choice(CUSTOMERS),
            random.randint(1, 10),
            random.randint(1, 5),
        ])
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["order_id", "customer_id", "product_id", "quantity"])
        w.writerows(rows)
    print(f"  [Generator] Batch {batch_num:04d} — {len(rows)} orders -> {os.path.basename(path)}")

def order_generator(n_batches=8, interval=4):
    for i in range(1, n_batches + 1):
        time.sleep(interval)
        write_order_batch(i)
    print("  [Generator] All batches written.")

print("Generator ready. Will write", 8, "batches x 8 orders = 64 total orders.")

## 4. Pipeline Stage 1 — Ingest & Enrich

Steps performed on the incoming file stream:

- **Read** new CSV files as they land (`maxFilesPerTrigger=1` — one file per micro-batch)
- **Add** `ingested_at` timestamp via `current_timestamp()`
- **Join** with the product catalog using a stream-static broadcast join
- **Compute** `line_total = quantity × unit_price` (rounded to 2 dp)
- **Flag** `is_high_value = line_total > 500` (e.g. Laptop orders)

In [ ]:
raw_orders = (
    spark.readStream
    .format("csv")
    .schema(order_schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", "1")
    .load(INPUT_DIR)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", input_file_name())
)

enriched_orders = (
    raw_orders
    .join(broadcast(products_df), on="product_id", how="inner")
    .withColumn("line_total",    spark_round(col("quantity") * col("unit_price"), 2))
    .withColumn("is_high_value", col("line_total") > 500.0)
    .select(
        "order_id", "customer_id", "product_id", "product_name",
        "category", "quantity", "unit_price", "line_total",
        "is_high_value", "ingested_at", "source_file"
    )
)

print("Enriched stream schema:")
enriched_orders.printSchema()

## 5. Pipeline Stage 2 — Windowed Revenue Aggregation

- **Watermark**: 10-second watermark on `ingested_at` bounds the window state kept in memory
- **Window**: 15-second tumbling windows — one bucket per [start, start+15s) per category
- **Output mode**: `update` — only updated windows emitted each trigger (vs `complete`
  which re-emits every window every trigger)
- Aggregations: `order_count`, `total_units`, `total_revenue`, `avg_order_value`

In [ ]:
windowed_revenue = (
    enriched_orders
    .withWatermark("ingested_at", "10 seconds")
    .groupBy(
        window(col("ingested_at"), "15 seconds"),
        col("category")
    )
    .agg(
        count("*").alias("order_count"),
        spark_sum("quantity").alias("total_units"),
        spark_round(spark_sum("line_total"), 2).alias("total_revenue"),
        spark_round(avg("line_total"), 2).alias("avg_order_value")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("category"),
        col("order_count"),
        col("total_units"),
        col("total_revenue"),
        col("avg_order_value")
    )
)

print("Windowed revenue schema:")
windowed_revenue.printSchema()

## 6. Start Both Sinks Simultaneously

Two concurrent streaming queries run on the enriched logical plan:

1. **Sink A — Parquet** (`append` mode): writes every enriched event record durably to Parquet files.
   Queryable with a batch `spark.read.parquet(...)` after the stream stops.
2. **Sink B — Memory** (`update` mode): writes windowed aggregates to an in-memory table named
   `revenue_dashboard`. Queryable with `spark.sql("SELECT * FROM revenue_dashboard")` while the
   stream is running.

Spark plans both queries together, sharing the file-source scan where possible.

In [ ]:
# Sink A — Parquet (raw enriched events)
q_parquet = (
    enriched_orders
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", PARQUET_OUT)
    .option("checkpointLocation", CKPT_ENRICH)
    .start()
)

# Sink B — Memory (windowed dashboard)
q_dashboard = (
    windowed_revenue
    .writeStream
    .format("memory")
    .queryName("revenue_dashboard")
    .outputMode("update")
    .option("checkpointLocation", CKPT_WINDOW)
    .start()
)

print("Both sinks started:")
print(f"  Parquet sink  id: {q_parquet.id}")
print(f"  Dashboard     id: {q_dashboard.id}")
print(f"Active queries: {len(spark.streams.active)}")

## 7. Run the Generator & Monitor Live

In [ ]:
gen_thread = threading.Thread(target=order_generator, args=(8, 4), daemon=True)
gen_thread.start()

print("Generator started. Live dashboard polls (every 8 seconds):\n")
for poll in range(5):
    time.sleep(8)
    print(f"=== Dashboard poll {poll+1} (t={(poll+1)*8}s) ===")
    dash = spark.sql("SELECT * FROM revenue_dashboard ORDER BY window_start, category")
    if dash.count() == 0:
        print("  (no data yet)")
    else:
        dash.show(20, truncate=False)

gen_thread.join(timeout=10)
print("\nGenerator finished.")

## 8. Stop Queries & Validate Parquet Output

In [ ]:
q_parquet.stop()
q_dashboard.stop()
print("Both queries stopped.")

# Read back the Parquet sink and validate
final_df = spark.read.parquet(PARQUET_OUT)
total_rows = final_df.count()
print(f"\nParquet sink validation:")
print(f"  Total enriched rows written: {total_rows}")

print("\nRows per category:")
final_df.groupBy("category").count().orderBy("category").show()

print("High-value orders (line_total > 500):")
final_df.filter(col("is_high_value")).select("order_id", "product_name", "quantity", "line_total").show()

print("Revenue by category (batch query on Parquet):")
(
    final_df
    .groupBy("category")
    .agg(
        count("*").alias("order_count"),
        spark_round(spark_sum("line_total"), 2).alias("total_revenue")
    )
    .orderBy("category")
    .show()
)

## 9. Final Dashboard Snapshot

In [ ]:
print("=== Final Revenue Dashboard (all windows) ===")
spark.sql("""
    SELECT window_start, window_end, category,
           order_count, total_units, total_revenue, avg_order_value
    FROM revenue_dashboard
    ORDER BY window_start, category
""").show(50, truncate=False)

print("\n=== Revenue Summary by Category ===")
spark.sql("""
    SELECT category,
           SUM(order_count)  AS total_orders,
           SUM(total_units)  AS total_units,
           ROUND(SUM(total_revenue), 2) AS total_revenue
    FROM revenue_dashboard
    GROUP BY category
    ORDER BY total_revenue DESC
""").show()

## 10. Pipeline Architecture Review

| Component | Detail |
|-----------|--------|
| **Source** | File source watching `INPUT_DIR` for new CSV files, one file per micro-batch |
| **Schema** | Explicit schema required — file source cannot infer schema in streaming mode |
| **Enrichment** | Stream-static broadcast join with the product catalog (reloaded once at startup) |
| **Computation** | `line_total = quantity × unit_price`, `is_high_value` boolean flag |
| **Sink A** | Parquet file sink — durable, queryable with batch Spark after the stream stops |
| **Sink B** | Memory sink — live `spark.sql()` dashboard queries during the stream run |
| **Watermark** | 10-second watermark on `ingested_at` bounds the window state held in memory |
| **Windows** | 15-second tumbling windows for per-category revenue roll-up |
| **Concurrency** | Two concurrent streaming queries on the same enriched logical plan |

Spark's Structured Streaming engine optimises the two queries together, sharing the file-source
scan where possible and managing separate checkpoints for each sink independently.

## Practice Exercises

1. Add a third sink using `foreachBatch` that prints a one-line summary to the console each
   micro-batch (e.g. batch id, row count, max line_total).
2. Replace the file source with the `rate` source and generate synthetic order data entirely
   in memory — no CSV files needed.
3. Add a `dropDuplicates(["order_id"])` step before enrichment to handle re-delivered files
   gracefully (idempotent ingestion).
4. Change the windowed aggregation to a **sliding window** (30s window, 10s slide) and compare
   result cardinality with the tumbling window version.
5. Add a second static table (customer names) and perform a second stream-static join to also
   enrich each order with the customer's name.

## Teardown

In [ ]:
shutil.rmtree(BASE, ignore_errors=True)
print("All temp directories cleaned up.")
spark.stop()
print("SparkSession stopped.")